In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go



/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [32]:
load_dotenv(override=True)
MODEL = "deepseek/deepseek-r1:free"
db_name = "vector_db"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key:
    print("OPENROUTER_API_KEY is set")
else:
    print("OPENROUTER_API_KEY is not set")


OPENROUTER_API_KEY is set


In [33]:
knowledge_base_path = "knowledge-base/**/*.md"

files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

Found 23 files in the knowledge base


In [34]:
entire_knowledge_base = ""

In [35]:
for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total number of characters in the knowledge base: {len(entire_knowledge_base)}")

Total number of characters in the knowledge base: 26955


In [39]:
import tiktoken

# 1. Try to get the model-specific encoding, otherwise fall back to a standard one
try:
    encoding = tiktoken.encoding_for_model(MODEL)
except KeyError:
    # Most modern models use the cl100k_base (GPT-4) or o200k_base (GPT-4o) patterns
    print(f"Warning: Model '{MODEL}' not recognized. Falling back to 'cl100k_base'.")
    encoding = tiktoken.get_encoding("cl100k_base")

# 2. Encode the knowledge base
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)

# 3. Use 'print' (printf is for C/C++)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for deepseek/deepseek-r1:free: 6,394


In [40]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 23 documents


In [42]:
documents[1]

Document(metadata={'source': 'knowledge-base/products/product2-secureops-fusion.md', 'doc_type': 'products'}, page_content='# SecureOps Fusion\n\n- Product ID: PROD-ACN-002\n- Company: Accenture\n- Category: Managed Cybersecurity Platform\n- Version: 4.1\n- Launch Year: 2022\n- Deployment: Managed Service, Hybrid Cloud\n- Primary Users: CISO Office, SOC Teams, IT Risk\n- Target Industries: Energy, Healthcare, Financial Services\n\n## Overview\nSecureOps Fusion unifies security monitoring, threat detection, and incident response orchestration into a single managed platform.\n\n## Core Features\n- 24x7 SOC telemetry aggregation and triage\n- Automated response playbooks for high-confidence alerts\n- Threat intelligence correlation and attack path analysis\n- Compliance reporting for common audit frameworks\n\n## Integrations\n- Splunk\n- Microsoft Sentinel\n- Palo Alto Cortex XSOAR\n- CrowdStrike Falcon\n\n## Pricing (Synthetic)\n- Base Managed SOC: USD 420,000 per year\n- Advanced Threa

In [43]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 41 chunks
First chunk:

page_content='# CareAssist Voice AI

- Product ID: PROD-ACN-003
- Company: Accenture
- Category: Healthcare Contact Center AI
- Version: 2.7
- Launch Year: 2024
- Deployment: Hybrid Cloud
- Primary Users: Patient Support Teams, Contact Center Managers
- Target Industries: Healthcare Providers, Payers

## Overview
CareAssist Voice AI is a conversational AI solution for patient engagement, appointment workflows, and claims routing in healthcare support centers.

## Core Features
- Natural language voice bots for inbound patient requests
- Intelligent triage and escalation to human agents
- Appointment booking and reminder workflows
- Sentiment analysis and quality scoring

## Integrations
- Epic
- Salesforce Health Cloud
- Genesys Cloud CX
- Twilio

## Pricing (Synthetic)
- Platform License: USD 210,000 per year
- Usage: USD 0.022 per voice minute
- Clinical Workflow Pack: USD 65,000 per year' metadata={'source': 'knowledge-base/products/product3-care

Document(metadata={'source': 'knowledge-base/products/product5-insightgrid-cx.md', 'doc_type': 'products'}, page_content='# InsightGrid CX\n\n- Product ID: PROD-ACN-005\n- Company: Accenture\n- Category: Customer Experience Analytics\n- Version: 1.9\n- Launch Year: 2025\n- Deployment: SaaS\n- Primary Users: Marketing Analytics, Customer Experience Leaders, Product Teams\n- Target Industries: Retail, Consumer Goods, Travel\n\n## Overview\nInsightGrid CX is an analytics product that combines customer interaction data, transaction behavior, and feedback signals to surface actionable experience insights.\n\n## Core Features\n- Unified customer journey analytics dashboard\n- Churn and loyalty propensity scoring\n- Real-time alerting for drop-off and conversion issues\n- Experiment tracking for UX and campaign changes\n\n## Integrations\n- Salesforce CRM\n- Adobe Experience Platform\n- Google Analytics 4\n- Zendesk\n\n## Pricing (Synthetic)\n- Professional: USD 145,000 per year\n- Enterprise